# V2.16 — PAD-UFES Lesion-Routing Enrichment Variant

**Issue:** [#157](https://github.com/revelainc/revela/issues/157)  
**Branch:** `Rehma_Revela_Dermascopic`  
**Date:** 2026-05-20  
**Author:** Rehma Aziz  
**Scope:** Dataset preparation only — no training, evaluation, or inference changes.

---

## Purpose

The Clinical V2 baseline has a lesion-routing false-negative count of **76**. Several enrichment variants from #152/#153 failed to reduce this and were not promoted. This notebook documents the preparation of a controlled PAD-UFES-20 enrichment variant targeting the lesion-routing class specifically.

PAD-UFES-20 is a clinical/macroscopic lesion dataset with `patient_id` and `lesion_id` metadata, enabling leakage-safe splitting. Only high-risk lesion labels (BCC, SCC, MEL, ACK) are included. Benign categories (NEV, SEK) are excluded to preserve the safety signal in the 5-class taxonomy.

**Output variant:** `clinical_v2_pad_ufes_lesion_enrichment`

## Setup

In [ ]:
import hashlib
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import GroupShuffleSplit

PAD_META  = "../../../Data Samples/pad-ufes-20_metadata_2026-05-15.csv"
IMAGE_DIR = "../data/PAD-UFES"

CLINv2_TRAIN = "../../../data/processed/clinical_v2/train.csv"
CLINv2_VAL   = "../../../data/processed/clinical_v2/val.csv"
CLINv2_TEST  = "../../../data/processed/clinical_v2/test.csv"

OUT_DIR = "../../../data/processed/clinical_v2_pad_ufes_lesion_enrichment"

SEED      = 42
VAL_RATIO = 0.15

DIAG3_TO_SHORT = {
    "Basal cell carcinoma":         "BCC",
    "Squamous cell carcinoma, NOS": "SCC",
    "Melanoma, NOS":                "MEL",
    "Solar or actinic keratosis":   "ACK",
    "Nevus":                        "NEV",
    "Seborrheic keratosis":         "SEK",
}

INCLUDE_MAP = {
    "BCC": "Lesion \u2014 dermoscopic review recommended",
    "SCC": "Lesion \u2014 dermoscopic review recommended",
    "MEL": "Lesion \u2014 dermoscopic review recommended",
    "ACK": "Lesion \u2014 dermoscopic review recommended",
}
EXCLUDE_LABELS = ["NEV", "SEK"]
EXCLUDE_REASON = {
    "NEV": "Benign melanocytic nevus \u2014 may dilute lesion-routing safety signal in 5-class taxonomy",
    "SEK": "Seborrheic keratosis \u2014 benign keratosis, more appropriate for 8-class taxonomy",
}

def file_hash(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()

print("Setup complete.")

## Section 1 — Load and Inspect PAD-UFES

In [ ]:
pad = pd.read_csv(PAD_META)
print(f"PAD-UFES total rows: {len(pad):,}")
print(f"Columns: {list(pad.columns)}")

REQUIRED_COLS = ["patient_id", "lesion_id", "diagnosis_3", "isic_id"]
for col in REQUIRED_COLS:
    assert col in pad.columns, f"Missing column: {col} — STOP"
print("\nAll required columns confirmed (patient_id, lesion_id, diagnosis_3, isic_id).")

pad["diagnostic"] = pad["diagnosis_3"].map(DIAG3_TO_SHORT)

print("\ndiagnosis_3 → short-code mapping:")
mapping_check = pad[["diagnosis_3", "diagnostic"]].drop_duplicates().dropna()
display(mapping_check.reset_index(drop=True))

print("\nAll diagnostic counts:")
display(pad["diagnostic"].value_counts().to_frame("count"))

## Section 2 — Taxonomy Mapping and Exclusions

In [ ]:
rows = []
for diag, count in pad["diagnostic"].value_counts().items():
    if pd.isna(diag):
        decision = "EXCLUDE"
        reason = "not in scope (unmapped diagnosis_3)"
    elif diag in INCLUDE_MAP:
        decision = "INCLUDE"
        reason = f"\u2192 {INCLUDE_MAP[diag]}"
    else:
        decision = "EXCLUDE"
        reason = EXCLUDE_REASON.get(diag, "not in scope")
    rows.append({"code": diag, "count": count, "decision": decision, "reason": reason})

breakdown = pd.DataFrame(rows)
display(breakdown)

pad_included = pad[pad["diagnostic"].isin(INCLUDE_MAP)].copy()
pad_excluded = pad[pad["diagnostic"].isin(EXCLUDE_LABELS)].copy()

pad_included["target_label"]   = pad_included["diagnostic"].map(INCLUDE_MAP)
pad_included["source_dataset"] = "pad_ufes_20"

print(f"\nIncluded rows: {len(pad_included):,}")
print(f"Excluded rows: {len(pad_excluded):,} (NEV: {(pad['diagnostic']=='NEV').sum()}, SEK: {(pad['diagnostic']=='SEK').sum()})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

include_counts = breakdown[breakdown["decision"] == "INCLUDE"].set_index("code")["count"]
exclude_counts = breakdown[breakdown["decision"] == "EXCLUDE"].set_index("code")["count"]

axes[0].bar(include_counts.index, include_counts.values, color="#2ecc71")
axes[0].set_title("Included labels (BCC/SCC/MEL/ACK)", fontsize=12)
axes[0].set_ylabel("Row count")
for i, v in enumerate(include_counts.values):
    axes[0].text(i, v + 5, str(v), ha="center", fontsize=10)

axes[1].bar(exclude_counts.index, exclude_counts.values, color="#e74c3c")
axes[1].set_title("Excluded labels (NEV/SEK)", fontsize=12)
axes[1].set_ylabel("Row count")
for i, v in enumerate(exclude_counts.values):
    axes[1].text(i, v + 5, str(v), ha="center", fontsize=10)

plt.suptitle("PAD-UFES-20 — Inclusion/Exclusion Breakdown", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Section 3 — Image Path Validation

In [ ]:
def resolve_image_path(isic_id):
    for ext in [".jpg", ".jpeg", ".png"]:
        p = os.path.join(IMAGE_DIR, isic_id + ext)
        if os.path.exists(p):
            return p
        p2 = os.path.join(IMAGE_DIR, isic_id)
        if os.path.exists(p2):
            return p2
    return None

pad_included["image_path"] = pad_included["isic_id"].apply(resolve_image_path)

missing = pad_included["image_path"].isna()
print(f"Image paths resolved: {(~missing).sum():,}/{len(pad_included):,}")
if missing.sum() > 0:
    print(f"WARNING: {missing.sum()} images not found — dropping rows.")
    print(pad_included[missing]["isic_id"].head(10).to_string())
    pad_included = pad_included[~missing].copy()
else:
    print("All image paths verified. No rows dropped.")

print(f"Final included rows after image check: {len(pad_included):,}")

## Section 4 — Patient-Level Leakage-Safe Split

In [ ]:
unique_patients = pad_included["patient_id"].unique()
print(f"Unique patients in included set: {len(unique_patients):,}")

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_RATIO, random_state=SEED)
train_idx, val_idx = next(gss.split(pad_included, groups=pad_included["patient_id"]))

pad_train = pad_included.iloc[train_idx].copy()
pad_val   = pad_included.iloc[val_idx].copy()

train_patients = set(pad_train["patient_id"])
val_patients   = set(pad_val["patient_id"])
assert len(train_patients & val_patients) == 0, "Patient leakage between train and val — STOP"

train_lesions = set(pad_train["lesion_id"])
val_lesions   = set(pad_val["lesion_id"])
assert len(train_lesions & val_lesions) == 0, "Lesion leakage between train and val — STOP"

print(f"PAD-UFES train rows: {len(pad_train):,} | val rows: {len(pad_val):,}")
print("Patient leakage: NONE confirmed")
print("Lesion leakage:  NONE confirmed")

val_counts = pad_val["target_label"].value_counts()
print(f"\nVal class distribution:")
display(val_counts.to_frame("count"))
if (val_counts < 3).any():
    print("WARNING: some val classes have < 3 examples. Documented — proceeding with patient-level split per issue scope.")

## Section 5 — Load Clinical V2 and Capture Test Hash

In [ ]:
test_hash_before = file_hash(CLINv2_TEST)
cv2_train = pd.read_csv(CLINv2_TRAIN)
cv2_val   = pd.read_csv(CLINv2_VAL)
cv2_test  = pd.read_csv(CLINv2_TEST)

print(f"Clinical V2 train: {len(cv2_train):,} | val: {len(cv2_val):,} | test: {len(cv2_test):,}")
print(f"Test hash (before): {test_hash_before}")

CV2_REQUIRED = list(cv2_train.columns)
print(f"\nClinical V2 columns ({len(CV2_REQUIRED)}):")
for c in CV2_REQUIRED:
    print(f"  {c}")

## Section 6 — Align PAD-UFES to Clinical V2 Schema and Merge

In [ ]:
LESION_CLASS_IDX = int(
    cv2_train[cv2_train["target_label"] == "Lesion \u2014 dermoscopic review recommended"]["class_idx"].iloc[0]
)
print(f"Lesion-routing class_idx in Clinical V2: {LESION_CLASS_IDX}")

for df in [pad_train, pad_val]:
    df["class_idx"]  = LESION_CLASS_IDX
    df["source_id"]  = df["isic_id"]
    df["case_id"]    = df["patient_id"]
    df["raw_label"]  = df["diagnosis_3"]
    for col in CV2_REQUIRED:
        if col not in df.columns:
            df[col] = None

pad_train_aligned = pad_train[CV2_REQUIRED].copy()
pad_val_aligned   = pad_val[CV2_REQUIRED].copy()

merged_train = pd.concat([cv2_train, pad_train_aligned], ignore_index=True)
merged_val   = pd.concat([cv2_val,   pad_val_aligned],   ignore_index=True)

print(f"Merged train: {len(merged_train):,} (+{len(pad_train_aligned):,} PAD-UFES rows)")
print(f"Merged val:   {len(merged_val):,} (+{len(pad_val_aligned):,} PAD-UFES rows)")

print("\nSource dataset breakdown in merged train:")
display(merged_train["source_dataset"].value_counts().to_frame("count"))

## Section 7 — Class Distribution Before and After

In [ ]:
before = cv2_train["target_label"].value_counts()
after  = merged_train["target_label"].value_counts()

dist = pd.DataFrame({
    "before_n":   before,
    "before_pct": (before / len(cv2_train) * 100).round(1),
    "after_n":    after,
    "after_pct":  (after / len(merged_train) * 100).round(1),
    "delta":      after - before,
}).fillna(0)
dist["delta"] = dist["delta"].astype(int)
dist.index.name = "target_label"

print("=== CLASS DISTRIBUTION BEFORE vs AFTER ENRICHMENT ===")
display(dist)

# Lesion-routing balance
label = "Lesion \u2014 dermoscopic review recommended"
cv2_lesion_n   = before[label]
cv2_lesion_pct = cv2_lesion_n / len(cv2_train) * 100
mrg_lesion_n   = after[label]
mrg_lesion_pct = mrg_lesion_n / len(merged_train) * 100

overweight = ("HIGH"     if mrg_lesion_pct > cv2_lesion_pct * 1.5
              else "MODERATE" if mrg_lesion_pct > cv2_lesion_pct * 1.2
              else "LOW")

print(f"\nLesion-routing rows:")
print(f"  Clinical V2 train:  {cv2_lesion_n:,} ({cv2_lesion_pct:.1f}%)")
print(f"  Merged train:       {mrg_lesion_n:,} ({mrg_lesion_pct:.1f}%)")
print(f"  Over-weighting risk: {overweight}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = [l[:30] + "..." if len(l) > 30 else l for l in before.index]

axes[0].barh(labels, before.values, color="#3498db")
axes[0].set_title("Before enrichment (Clinical V2 train)", fontsize=11)
axes[0].set_xlabel("Row count")
for i, v in enumerate(before.values):
    axes[0].text(v + 20, i, f"{v:,}", va="center", fontsize=9)

after_sorted = after.reindex(before.index)
colors = ["#e74c3c" if l == label else "#3498db" for l in before.index]
axes[1].barh(labels, after_sorted.values, color=colors)
axes[1].set_title("After enrichment (merged train)", fontsize=11)
axes[1].set_xlabel("Row count")
for i, v in enumerate(after_sorted.values):
    axes[1].text(v + 20, i, f"{v:,}", va="center", fontsize=9)

axes[1].legend(handles=[
    plt.Rectangle((0,0),1,1, color="#e74c3c", label="Lesion (enriched)"),
    plt.Rectangle((0,0),1,1, color="#3498db", label="Other classes"),
], loc="lower right")

plt.suptitle("Class Distribution Before vs After PAD-UFES Enrichment", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Section 8 — Final Asserts and Save

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

assert list(merged_train.columns) == CV2_REQUIRED, "Column mismatch in train — STOP"
assert list(merged_val.columns)   == CV2_REQUIRED, "Column mismatch in val — STOP"
assert merged_train["source_dataset"].isna().sum() == 0, "Missing source_dataset in train — STOP"
assert merged_val["source_dataset"].isna().sum() == 0,   "Missing source_dataset in val — STOP"

# Validate image paths for PAD-UFES rows only (Clinical V2 rows are cloud-stored, pre-validated)
for df_name, pad_df in [("train", pad_train_aligned), ("val", pad_val_aligned)]:
    missing_paths = pad_df[~pad_df["image_path"].apply(os.path.exists)]
    assert len(missing_paths) == 0, f"{df_name} PAD-UFES: {len(missing_paths)} missing image paths — STOP"
    print(f"{df_name} PAD-UFES: all {len(pad_df):,} image paths verified")

merged_train.to_csv(f"{OUT_DIR}/train.csv", index=False)
merged_val.to_csv(  f"{OUT_DIR}/val.csv",   index=False)
cv2_test.to_csv(    f"{OUT_DIR}/test.csv",  index=False)

test_hash_after = file_hash(CLINv2_TEST)
assert test_hash_before == test_hash_after, "Clinical V2 test file was modified — STOP"
print(f"Test set hash: {test_hash_after} — UNCHANGED")
print(f"\nSaved to: {OUT_DIR}/")
print(f"  train.csv: {len(merged_train):,} rows")
print(f"  val.csv:   {len(merged_val):,} rows")
print(f"  test.csv:  {len(cv2_test):,} rows (Clinical V2 test, unmodified)")

## Section 9 — Write Config YAML

In [ ]:
config = {
    "variant_name":       "clinical_v2_pad_ufes_lesion_enrichment",
    "base_variant":       "clinical_v2",
    "enrichment_source":  "PAD-UFES-20",
    "enrichment_purpose": "Reduce lesion-routing false negatives (baseline FN: 76)",
    "taxonomy":           "5-class Clinical V2 \u2014 unchanged",
    "pad_ufes_mapping": {
        "BCC": "Lesion \u2014 dermoscopic review recommended",
        "SCC": "Lesion \u2014 dermoscopic review recommended",
        "MEL": "Lesion \u2014 dermoscopic review recommended",
        "ACK": "Lesion \u2014 dermoscopic review recommended",
    },
    "excluded_labels": {
        "NEV": "Benign nevus \u2014 may dilute lesion-routing signal",
        "SEK": "Seborrheic keratosis \u2014 more appropriate for 8-class taxonomy",
    },
    "split_strategy": "patient_level",
    "split_seed":     SEED,
    "val_ratio":      VAL_RATIO,
    "data": {
        "train": f"{OUT_DIR}/train.csv",
        "val":   f"{OUT_DIR}/val.csv",
        "test":  f"{OUT_DIR}/test.csv",
    },
    "promotion_criteria": {
        "lesion_routing_fn_max":    76,
        "macro_f1_min":             0.6420,
        "scin_macro_f1_min":        0.4028,
        "fitzpatrick_macro_f1_min": 0.6366,
        "inflammatory_f1_note":     "must not collapse",
    },
    "out_of_scope": [
        "Model training",
        "Model evaluation",
        "Model promotion",
        "8-class taxonomy",
        "Clinical-readiness claims",
    ],
}

os.makedirs("../../../config", exist_ok=True)
config_path = "../../../config/clinical_v2_pad_ufes_lesion_enrichment_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config, f, allow_unicode=True, sort_keys=False)

print(f"Saved: {config_path}")
print("\nConfig contents:")
print(yaml.dump(config, allow_unicode=True, sort_keys=False))

## Summary

| Item | Value |
|---|---|
| PAD-UFES total rows | 2,298 |
| Included (BCC+SCC+MEL+ACK) | 1,819 |
| Excluded (NEV) | 244 |
| Excluded (SEK) | 235 |
| PAD-UFES train rows | 1,556 |
| PAD-UFES val rows | 263 |
| Clinical V2 train (before) | 6,986 |
| Merged train (after) | 8,542 |
| Clinical V2 val (before) | 1,518 |
| Merged val (after) | 1,781 |
| Test set rows | 1,515 (unchanged) |
| Lesion-routing rows before | 1,739 (24.9%) |
| Lesion-routing rows after | 3,295 (38.6%) |
| Over-weighting risk | **HIGH** |
| Patient leakage | NONE |
| Lesion leakage | NONE |
| PAD-UFES image paths verified | 1,819 / 1,819 |
| Test hash unchanged | ✓ `4b510381927f6265446a62cb990e69fd` |

### Promotion Criteria (future training task)

| Metric | Threshold |
|---|---|
| Lesion-routing FN | < 76 |
| Combined macro-F1 | ≥ 0.6420 |
| SCIN macro-F1 | ≥ 0.4028 |
| Fitzpatrick17k macro-F1 | ≥ 0.6366 |
| Inflammatory-class F1 | must not collapse |

### Out of Scope
- Model training, evaluation, or promotion
- 8-class taxonomy
- App/inference changes
- Clinical-readiness claims